환경변수

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

라이브러리

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

LLM

In [3]:
llm = ChatOpenAI(
    model="gpt-5.4-nano",
    temperature=0
)

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

Vector DB 연결

In [4]:
law_db = Chroma(
    persist_directory="vector_db/laws",
    embedding_function=embedding
)

precedent_db = Chroma(
    persist_directory="vector_db/precedents",
    embedding_function=embedding
)

qna_db = Chroma(
    persist_directory="vector_db/qna",
    embedding_function=embedding
)

State

In [5]:
class GraphState(TypedDict):
    
    question: str

    qna_docs: list
    issue_summary: str

    law_docs: list
    law_analysis: str

    precedent_docs: list
    precedent_analysis: str

    final_answer: str

질의회시 검색 노드

In [6]:
def retrieve_qna_node(state):

    question = state["question"]

    docs = qna_db.similarity_search(
        question,
        k=10
    )

    print("\n===== QNA(질의회시) RETRIEVAL TOP5 =====")
    for i, doc in enumerate(docs, 1):
        print(f"\n[QNA {i}]")
        print("metadata title:", doc.metadata.get("title", ""))
        print("질의(임베딩 대상):", doc.page_content)

    return {
        "qna_docs": docs
    }

쟁점 추출 노드 (질의회시 기반)

In [7]:
def extract_issue_node(state):

    question = state["question"]

    qna_context_parts = []
    for doc in state["qna_docs"]:
        m = doc.metadata
        title = m.get("title", "")
        chapter = m.get("chapter_title", "")
        ref_no = m.get("ref_no", "")
        ref_date = m.get("ref_date", "")
        full_content = m.get("full_content", doc.page_content)

        label = f"[{chapter} / {title} / {ref_no}, {ref_date}]"
        qna_context_parts.append(f"{label}\n{full_content[:600]}")

    qna_context = "\n\n".join(qna_context_parts)

    prompt = f"""
다음은 사용자의 사례와, 이와 유사한 고용노동부 질의회시(행정해석) 사례입니다.

사용자 사례:
{question}

유사 질의회시 사례:
{qna_context}

작업:
위 질의회시 사례들을 참고하여, 사용자 사례의 핵심 법률 쟁점을
법령/판례 검색에 사용할 수 있는 법률 용어 형태로 정리하세요.

출력 형식 (반드시 준수):
쟁점: (한 문장, 법률 용어 중심)
관련 법률 분야: (예: 임금, 근로시간, 근로계약, 취업규칙, 재해보상, 총칙 중 해당 항목)
검색 키워드: (법령/판례 검색에 쓸 키워드, 10단어 이내, 쉼표로 구분)
"""

    issue_summary = llm.invoke(prompt).content.strip()

    print("\n===== EXTRACTED ISSUE =====")
    print(issue_summary)

    return {
        "issue_summary": issue_summary
    }

법령 검색 노드

In [8]:
def retrieve_law_node(state):

    question = state["question"]
    issue_summary = state.get("issue_summary", "")

    search_query = f"{question}\n\n{issue_summary}" if issue_summary else question

    docs = law_db.similarity_search(
        search_query,
        k=5
    )

    print("\n===== LAW RETRIEVAL TOP5 =====")
    for i, doc in enumerate(docs, 1):
        print(f"\n[LAW {i}]")
        print("metadata:", doc.metadata)
        print(doc.page_content[:500])  # 앞 500자만 출력

    return {
        "law_docs": docs
    }

법령 분석 노드

In [9]:
def analyze_law_node(state):

    question = state["question"]

    # 조문 단위 메타데이터로 출처 라벨 구성
    law_context_parts = []
    for doc in state["law_docs"]:
        m = doc.metadata
        law_name   = m.get("law_name", "")
        article_no = m.get("article_no", "")
        article_title = m.get("article_title", "")
        chapter    = m.get("chapter", "")
        source_pdf = m.get("source_pdf", "")
        page       = m.get("page", "")

        # "근로기준법 / 제3장 제43조 (임금 지급) / 1p" 형태
        parts = [p for p in [law_name, chapter, f"{article_no} ({article_title})" if article_title else article_no] if p]
        label = " / ".join(parts)
        if page:
            label += f" / {page}p"
        if source_pdf:
            label += f" [{source_pdf}]"

        law_context_parts.append(f"[출처: {label}]\n{doc.page_content}")

    law_context = "\n\n".join(law_context_parts)

    prompt = f"""
당신은 대한민국 노동법 전문 AI입니다.

사용자 사례:
{question}

관련 법령 (각 조문 앞에 출처 표시):
{law_context}

규칙

1. 관련 법률만 설명
2. 위법이라고 단정 금지
3. 반드시 법률명, 장(章), 조(條), 항(項)을 명시 (예: 근로기준법 제3장 제43조 제1항)
4. 조문이 확인되지 않으면 "조문 불명확"으로 표시
5. 출처 PDF 파일명도 함께 표시

출력 형식 (반드시 준수)

관련 법률:
- 법률명 제X장 제X조 제X항 (출처: [파일명] Xp): 조항 내용 요약

설명:
- 해당 조항이 사용자 사례에 적용되는 이유
"""

    result = llm.invoke(prompt).content

    return {
        "law_analysis": result
    }


판례 검색 노드

In [10]:
def retrieve_precedent_node(state):

    keyword_prompt = f"""
다음 사용자 사례와 법률 분석에서
판례 검색에 사용할 핵심 법률 쟁점 키워드를
10단어 이내로 추출하세요.

사용자 사례: {state['question']}
추출된 쟁점: {state.get('issue_summary', '')}
법률 분석 요약: {state['law_analysis'][:300]}

출력: 키워드만, 문장 금지
"""

    search_query = llm.invoke(keyword_prompt).content.strip()

    print("\n===== PRECEDENT SEARCH QUERY =====")
    print(search_query)

    docs = precedent_db.similarity_search(
        search_query,
        k=5
    )

    print("\n===== PRECEDENT RETRIEVAL TOP5 =====")
    for i, doc in enumerate(docs, 1):
        print(f"\n[PRECEDENT {i}]")
        print("metadata:", doc.metadata)
        print(doc.page_content[:500])

    return {
        "precedent_docs": docs
    }

판례 분석 노드

In [11]:
def analyze_precedent_node(state):

    # 판례 메타데이터: source_file (예: "2011다23149.md")
    precedent_context_parts = []
    for doc in state["precedent_docs"]:
        m = doc.metadata
        source_file = m.get("source_file", "알 수 없는 판례")
        # 파일명에서 사건번호 추출 (확장자 제거)
        case_no = source_file.replace(".md", "").replace(".json", "")
        label = f"[판례: {case_no}]"
        precedent_context_parts.append(f"{label}\n{doc.page_content}")

    precedent_context = "\n\n".join(precedent_context_parts)

    prompt = f"""
사용자 사례:
{state['question']}

검색된 판례 (각 판례 앞에 출처 표시):
{precedent_context}

규칙

1. 반드시 판례명(사건번호) 전체 표기 (예: 대법원 2019. 4. 18. 선고 2016다2451 판결)
2. 판결 요지 1~2문장으로 핵심 요약
3. 사용자 사례와의 유사점을 구체적으로 설명
4. 법률 자문처럼 단정 금지
5. 출처 파일명(사건번호) 반드시 표시

출력 형식 (반드시 준수)

판례명: [전체 사건번호 포함]
출처: [파일명]
판결 요지: [핵심 내용 1~2문장]
사용자 사례 관련성: [구체적 유사점]
"""

    result = llm.invoke(prompt).content

    return {
        "precedent_analysis": result
    }


최종 답변 노드

In [12]:
def generate_answer_node(state):

    qna_context_parts = []
    for doc in state.get("qna_docs", []):
        m = doc.metadata
        label = f"[{m.get('title','')} / {m.get('ref_no','')}, {m.get('ref_date','')}]"
        full_content = m.get("full_content", doc.page_content)
        qna_context_parts.append(f"{label}\n{full_content[:400]}")
    qna_context = "\n\n".join(qna_context_parts)

    prompt = f"""
사용자 사례:
{state["question"]}

추출된 법률 쟁점:
{state.get("issue_summary", "")}

법률 분석:
{state["law_analysis"]}

판례 분석:
{state["precedent_analysis"]}

참고 행정해석(질의회시):
{qna_context}

당신은 노동법 분석 시스템이다.

반드시 아래 형식만 출력하라.

# 관련 법률

- 법률명 및 조항: 예) 근로기준법 제3장 제43조 제1항
- 조항 내용 요약
- 적용 가능성

# 관련 판례

- 판례명: 예) 대법원 2019. 4. 18. 선고 2016다2451 판결
- 출처: 판례 파일명 / 분류
- 판결 요지
- 사용자 사례와의 관련성

# 관련 행정해석 (참고용, 법적 구속력 없음)

- 제목 및 문서번호 (예: 근로기준정책과-1384, 2021.5.11.)
- 회시 요지
- 사용자 사례와의 관련성

# 결론

- 현재 정보 기준 분석 결과
- 주요 근거 조항 재정리 (법률명 + 조번호)

규칙

1. 추가 질문 금지
2. 추가 정보 요청 금지
3. 상담 권유 금지
4. "원하시면", "추가로 알려주시면" 같은 문장 금지
5. 위 형식 외 내용 출력 금지
6. 최종 결과만 작성
7. 조항 번호가 불명확한 경우 "조항 확인 필요"로 표시
8. 행정해석은 법령/판례와 명확히 구분하여 "참고용"으로만 제시
"""

    answer = llm.invoke(prompt).content

    return {
        "final_answer": answer
    }

Graph 생성

In [13]:
builder = StateGraph(GraphState)

builder.add_node(
    "retrieve_qna",
    retrieve_qna_node
)

builder.add_node(
    "extract_issue",
    extract_issue_node
)

builder.add_node(
    "retrieve_law",
    retrieve_law_node
)

builder.add_node(
    "analyze_law",
    analyze_law_node
)

builder.add_node(
    "retrieve_precedent",
    retrieve_precedent_node
)

builder.add_node(
    "analyze_precedent",
    analyze_precedent_node
)

builder.add_node(
    "generate_answer",
    generate_answer_node
)

Edge 연결

In [14]:
builder.set_entry_point(
    "retrieve_qna"
)

builder.add_edge(
    "retrieve_qna",
    "extract_issue"
)

builder.add_edge(
    "extract_issue",
    "retrieve_law"
)

builder.add_edge(
    "retrieve_law",
    "analyze_law"
)

builder.add_edge(
    "analyze_law",
    "retrieve_precedent"
)

builder.add_edge(
    "retrieve_precedent",
    "analyze_precedent"
)

builder.add_edge(
    "analyze_precedent",
    "generate_answer"
)

builder.add_edge(
    "generate_answer",
    END
)

graph = builder.compile()

법령 PDF 벡터DB 생성 (조문 단위 파싱 — law_parser.py 사용)

In [15]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_chroma import Chroma

# ==========================================================
# 법령 JSON 로드
# ==========================================================

law_root = Path("data/process/법률")

if not law_root.exists():
    raise FileNotFoundError(
        f"폴더가 없습니다: {law_root.resolve()}\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

json_files = sorted(law_root.rglob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"{law_root} 안에 JSON 파일이 없습니다.\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

print(f"JSON 파일 {len(json_files)}개 발견")

# ==========================================================
# JSON → Document
# ==========================================================

all_law_docs = []

for json_file in json_files:

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    if isinstance(items, dict):
        items = [items]

    for item in items:

        meta = item.get("metadata", {})

        chapter_title = meta.get("chapter_title", "")
        article_title = meta.get("article_title", "")
        page_content = item.get("page_content", "")

        # --------------------------------------------------
        # 임베딩용 텍스트
        # chapter_title
        # article_title
        # page_content
        # --------------------------------------------------

        embedding_text = f"""
{chapter_title}

{article_title}

{page_content}
"""

        embedding_text = " ".join(
            embedding_text.split()
        )

        all_law_docs.append(
            Document(
                page_content=embedding_text,
                metadata=meta
            )
        )

if not all_law_docs:
    raise ValueError(
        "로드된 법령 Document가 0개입니다."
    )

print(f"\n총 {len(all_law_docs)}개 법령 Document 로드 완료")

print("\n========== Document 예시 ==========")
print(all_law_docs[0].page_content[:500])

print("\n========== Metadata 예시 ==========")
for k, v in all_law_docs[0].metadata.items():
    print(f"{k}: {v}")

# ==========================================================
# Chroma DB 생성
# ==========================================================

law_db = Chroma.from_documents(
    documents=all_law_docs,
    embedding=embedding,
    persist_directory="vector_db/laws"
)

print("\n법령 DB 생성 완료")
print("저장 위치: vector_db/laws")

JSON 파일 24개 발견
로딩 중: 근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).json
로딩 중: 근로기준법 시행령(대통령령)(제35436호)(20251023).json
로딩 중: 근로기준법(법률)(제20520호)(20251023) (1).json
로딩 중: 근로자퇴직급여 보장법 시행규칙(고용노동부령)(제00359호)(20220712).json
로딩 중: 근로자퇴직급여 보장법 시행령(대통령령)(제36220호)(20260324).json
로딩 중: 근로자퇴직급여 보장법(법률)(제21135호)(20251111).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률 시행규칙(노동부령)(제00277호)(20070701).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률 시행령(대통령령)(제31611호)(20210408).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률(법률)(제18177호)(20210518).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행규칙(고용노동부령)(제00453호)(20251001).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령(대통령령)(제35806호)(20251001).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률(법률)(제21065호)(20251001).json
로딩 중: 노동조합 및 노동관계조정법 시행규칙(고용노동부령)(제00463호)(20260310).json
로딩 중: 노동조합 및 노동관계조정법 시행령(대통령령)(제36159호)(20260310).json
로딩 중: 노동조합 및 노동관계조정법(법률)(제21045호)(20260310).json
로딩 중: 산업안전보건법 시행규칙(고용노동부령)(제00470호)(20260601).json
로딩 중: 산업안전보건법 시행령(대통령령)(제36220호)(20260324).json
로딩 중: 산업안전보건법(법률)(제21374호)(20260601).json
로딩 중:

판례 벡터DB 생성

In [16]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_chroma import Chroma

precedent_root = Path("data/process/판례")
all_docs = []

# JSON 파일로 저장된 판례 로드
for json_file in sorted(precedent_root.rglob("*.json")):

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    # 파일 하나에 판례 여러 건이 리스트로 담겨 있을 수 있음
    if isinstance(items, dict):
        items = [items]

    for item in items:
        doc = Document(
            page_content=item["page_content"],
            metadata=item.get("metadata", {})
        )
        all_docs.append(doc)

print(f"\n총 {len(all_docs)}개 판례 Document 로드 완료")
if all_docs:
    print("[메타데이터 예시]", all_docs[0].metadata)

# 판례는 1건이 하나의 Document — 청크 분할 없음
precedent_db = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding,
    persist_directory="vector_db/precedents"
)

print("판례 DB 생성 완료")

로딩 중: 2011다23149.json
로딩 중: 2006다64245.json
로딩 중: 2014다44673.json
로딩 중: 2018도6486.json
로딩 중: 2001도204.json
로딩 중: 2017도4343.json
로딩 중: 2022도2188.json
로딩 중: 98도1260.json
로딩 중: 2010다91046.json
로딩 중: 2012다89399.json
로딩 중: 2015다73067.json
로딩 중: 89다카2292.json
로딩 중: 94다19501.json
로딩 중: 2022두64518.json
로딩 중: 90누2772.json
로딩 중: 97다5015.json
로딩 중: 2012다106423.json
로딩 중: 2015다73067.json
로딩 중: 2020도15393.json
로딩 중: 2008다57852.json
로딩 중: 2008다6052.json
로딩 중: 96다24699.json
로딩 중: 2007다90760.json
로딩 중: 2016다255910.json
로딩 중: 93다26168.json
로딩 중: 97누18189.json
로딩 중: 2002다11458.json
로딩 중: 90다11554.json
로딩 중: 91다36666.json
로딩 중: 2020도16541.json
로딩 중: 2004다29736.json
로딩 중: 2006도777.json
로딩 중: 2009다51417.json
로딩 중: 2002다62432.json
로딩 중: 2019두55859.json
로딩 중: 2006두4912.json
로딩 중: 2017두45933.json
로딩 중: 2019두62604.json
로딩 중: 2020다270503.json
로딩 중: 2007두22498.json
로딩 중: 2017두74702.json
로딩 중: 2012두4852.json
로딩 중: 2015두51651.json
로딩 중: 2017두76005.json
로딩 중: 2018두47264.json
로딩 중: 2019두38571.json

총 46개 판례 Document

질의회시집 벡터DB 생성

In [17]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_chroma import Chroma

qna_root = Path("data/process/질의회시집")
qna_docs = []

# processer_qna.py 출력 구조:
#   page_content        -> "질의: <질의 내용>" (임베딩/검색용, 회시 본문 제외)
#   metadata.full_content -> "질의: ...\n\n회시: ..." (분석/표시용 전체 내용)
for json_file in sorted(qna_root.rglob("*.json")):

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    # 파일 하나에 질의회시 여러 건이 리스트로 담겨 있을 수 있음
    if isinstance(items, dict):
        items = [items]

    for item in items:
        doc = Document(
            page_content=item["page_content"],       # "질의: ..." 만 임베딩
            metadata=item.get("metadata", {})         # full_content 포함
        )
        qna_docs.append(doc)

print(f"\n총 {len(qna_docs)}개 질의회시 Document 로드 완료")
if qna_docs:
    print("[임베딩 대상 예시]", qna_docs[0].page_content)
    print("[메타데이터 키]", list(qna_docs[0].metadata.keys()))

# 질의회시는 1건이 하나의 Document — 청크 분할 없음
qna_db = Chroma.from_documents(
    documents=qna_docs,
    embedding=embedding,
    persist_directory="vector_db/qna"
)

print("질의회시 DB 생성 완료")

로딩 중: 질의회시집.json

총 397개 질의회시 Document 로드 완료
[임베딩 대상 예시] 질의: 「군형법」 제2조제2항의 군인(장교, 준사관, 부사관, 병)이 「근로기준법」 제2조 제1항제1호에 따른 근로자인지
[메타데이터 키] ['title', 'answer', 'chapter_num', 'chapter_name', 'reference', 'ref_date', 'start_page', 'source', 'title_keywords']
질의회시 DB 생성 완료


In [25]:
result = graph.invoke(
    {
        "question":
        """상금자가 업무적으로 지시한 사항에 대해서 저의 실수가 있었습니다. 이를 이유로 약간의 위협을 당했는데 직장내 폭행으로 볼 수 있나요?"""
    }
)

print(result["final_answer"])


===== QNA(질의회시) RETRIEVAL TOP5 =====

[QNA 1]
metadata title: 「근로기준법」 제76조의3 제6항 직장 내 괴롭힘 신고로 인한 불리한
질의(임베딩 대상): 질의: 처우 관련 정당한 업무지시 불응 등에 의한 것으로 징계사유가 존재함에도, 「근로기준법」 제76조의3 제6항에 따라 직장 내 괴롭힘 신고를 하였다는 사유로 어떠한 해고 등 처분을 할 수 없는지

[QNA 2]
metadata title: 회사의 전직금지 가처분 신청이 취업방해에 해당할 수 있는지
질의(임베딩 대상): 질의: 회사가 퇴직한 근로자를 대상으로 법원에 전직금지 가처분 신청을 한 경우 「근로기준법」에서 금지하는 취업 방해에 해당하는지

[QNA 3]
metadata title: 사용자가 아닌 근로자가 직장 내 괴롭힘 행위자일 경우 처벌 가능 여부
질의(임베딩 대상): 질의: 근로자인 직장 내 괴롭힘 행위자에 대한 처벌이 가능한지 구체적인 사실관계를 확인할 수 없어 귀하의 민원에 대해 명확하게 답변 드리기는 어려우나, 직장 내 괴롭힘 행위자가 사용자(사용자의 「민법」 제767조에 따른 친족 중 1)사용자의 배우자, 2)사용자의 4촌 이내의 혈족, 3)사용자의 4촌 이내의 인척이 해당 사업 또는 사업장의 근로자인 경우를 포함)인 경우 「근로기준법」 제76조의2 위반에 대하여 1천만원 이하의 과태료를 부과하도록

[QNA 4]
metadata title: 노동위원회 구제명령에 따른 임금상당액의 임금성 여부
질의(임베딩 대상): 질의: 노동위원회의 부당해고 구제명령으로 사용자에게 지급의무가 발생한 임금 상당액이 「근로기준법」상 임금에 해당하지 않는다고 판단한 우리부 행정 해석에 변경사항이 있는지

[QNA 5]
metadata title: 지정변제로 인정되는 명시적 의사표시
질의(임베딩 대상): 질의: 임금채권의 일부를 변제하는 경우 민법상 지정변제로 판단될 수 있는 채무자의 명시적 의사표시 귀 질의만으로 구체적인 사실관계를 확인할 수 없어 명확한 답

In [23]:
from IPython.display import Markdown, display

display(
    Markdown(result["final_answer"])
)

# 관련 법률

- 법률명 및 조항: 근로기준법 제44조의2 제1항
- 조항 내용 요약: 건설업에서 사업이 2차례 이상 도급으로 이루어진 경우, 직상 수급인이 하수급인이 체불한 “해당 건설공사에서 발생한 임금”을 지급하지 못하게 되면 직상 수급인은 하수급인과 **연대하여** 임금 지급 책임을 짐.
- 적용 가능성: 화물차 일용직이라도 **해당 운송이 건설공사와 결합된 도급구조(건설업 2차 이상 도급)**로 평가되고, 체불이 하수급인 임금 체불로 보이는 경우 **책임 주체를 직상 수급인(연대책임 가능)**까지 확장하는 데 적용될 수 있음.

- 법률명 및 조항: 근로기준법 제44조의2 제2항
- 조항 내용 요약: 직상 수급인이 “건설사업자”가 아닌 경우, 상위 수급인 중 **최하위의 건설사업자**를 직상 수급인으로 보아 연대책임 구조를 정함.
- 적용 가능성: 직상 수급인/중간 단계가 건설사업자가 아닌 형태인 경우, **실질적으로 연대책임 지는 ‘직상 수급인’ 범위**를 정하는 데 활용될 수 있음(다만 사안이 건설 도급 구조일 때).

- 법률명 및 조항: 근로기준법 제43조의6 제1항
- 조항 내용 요약: 임금체불 관련하여 고용노동부장관이 체불사업주 및 체불근로자 등 관련 자료를 관계기관에 요청할 수 있음.
- 적용 가능성: “누가 임금을 줘야 하는지(책임 주체)”를 다투기 위해 **체불 관련 실체/거래관계/사업주를 특정하는 자료확보 단계**에서 간접적으로 유의미함(책임 주체 확정 조문 자체는 아님).

- 법률명 및 조항: 근로기준법 제43조의6 제3항
- 조항 내용 요약: 요청을 받은 자는 정당한 사유가 없으면 요청에 따라야 함.
- 적용 가능성: 임금체불 입증 및 책임자 특정에 필요한 자료 제공을 강제/지원하는 기능.

- 법률명 및 조항: 근로기준법 시행령 제2조 제1항
- 조항 내용 요약: 평균임금 산정기간에서 제외되는 기간(사용자 귀책사유로 휴업 등) 및 산정 기준 총액 제외 규정을 정함.
- 적용 가능성: 화물차 일용직이라도 체불이 발생한 후 **평균임금 산정 관련 쟁점(휴업수당 등)**이 생기면 적용될 수 있으나, “임금 지급 책임의 주체” 자체를 정하는 규정은 아님.

# 관련 판례

- 판례명: 대법원 2010. 5. 20. 선고 2007다90760 전원합의체 판결
- 출처: 2007다90760.txt / 분류: 퇴직금 분할약정 무효·부당이득·임금상계 제한 등
- 판결 요지: 퇴직금 분할약정이 중간정산 요건을 충족하지 못하면 퇴직금청구권을 사전에 포기하는 형태로 원칙적으로 무효가 될 수 있고, 무효로 지급된 퇴직금 명목 금원은 법률상 원인 없는 부당이득이 될 수 있음. 사용자는 임금전액지급 원칙 때문에 상계가 제한되며, 예외적으로 부당이득반환채권을 자동채권으로 상계하는 경우에도 퇴직금채권의 압류금지 한도 등 문제가 함께 다뤄짐.
- 사용자 사례와의 관련성: 화물차 일용직의 “사용자(고용주)에게 지급의무가 귀속되는지” 논점에서, 결국 임금/퇴직금 성격의 금전 지급 문제는 **사용자 책임**을 전제로 판단틀이 형성된다는 점에 참고가 됨(다만 사안이 임금체불로서의 ‘임금 직접불 원칙/사용자 개념’ 그 자체를 직접 판시한 것은 아님).

- 판례명: 대법원 2013. 12. 18. 선고 2012다89399 전원합의체 판결
- 출처: 2012다89399.txt / 분류: 통상임금 해당 여부 판단기준
- 판결 요지: 통상임금 해당 여부는 임금 명칭보다 실제 성질(소정근로의 대가로서 정기적·일률적·고정적으로 지급되는지 등)로 객관적으로 판단하며, 통상임금에서 제외하기로 한 노사합의는 원칙적으로 효력이 없음(신의칙 예외 논의는 별도).
- 사용자 사례와의 관련성: 화물차 일용직에서도 지급된 금원이 임금(통상임금 산정 기초)으로 평가될 경우 **사용자에게 지급의무가 귀속**되는 방향으로 분쟁이 정리될 수 있다는 점에서 관련성 있음(다만 지급 책임의 ‘주체’ 특정 규정 판시는 아님).

- 판례명: 대법원 2014. 6. 19. 선고 2014다44673 판결
- 출처: 2014다44673.txt / 분류: 최저임금 산입/주휴수당 등 비교대상 임금
- 판결 요지: 최저임금 미달 여부 판단에서 최저임금에 산입하지 않는 임금을 제외하고 비교대상 임금과 비교하며, 유급휴일 주휴수당은 비교대상 임금에 포함됨.
- 사용자 사례와의 관련성: 화물차 일용직의 임금 구성에서 특정 항목이 임금 범위/산정에 포함되는지(결국 지급의무 귀속은 사용자) 판단할 때 참고가 됨.

- 판례명: 대법원 2011. 7. 14. 선고 2011다23149 판결
- 출처: 2011다23149.txt / 분류: 평균임금 산정 임금총액 포함 범위
- 판결 요지: 평균임금 기초가 되는 임금총액에는 명칭과 무관하게 사용자가 근로의 대상으로 근로자에게 계속적·정기적으로 지급하고 지급의무가 사용자에게 지워지는 경우 포함됨.
- 사용자 사례와의 관련성: 화물차 일용직이라도 정기·계속적 성격의 금원이 있으면 평균임금(및 관련 계산)의 기초가 사용자에게 귀속되어 지급범위가 달라질 수 있다는 점에서 관련성 있음.

- 판례명: 대법원 2009. 12. 10. 선고 2008다57852 판결
- 출처: 2008다57852.txt / 분류: 포괄임금제 성립 요건
- 판결 요지: 포괄임금제가 성립하려면 단체협약/임금협정 내용 등 포괄임금 구조 해당성 등을 종합 판단해야 하고, 임금 항목 분리·산정이 명확하면 포괄임금 단정이 어렵다. 통상임금 산입을 배제하는 합의는 무효가 될 수 있음.
- 사용자 사례와의 관련성: 일용 운송에서도 “일당에 포함됐다/따로 수당 없다” 식의 주장이 통용되기 어려울 수 있음을 보여주며, 결과적으로 사용자에게 임금 지급 의무가 문제될 수 있다는 점에서 간접 관련성 있음.

# 관련 행정해석 (참고용, 법적 구속력 없음)

- 제목 및 문서번호: 임금의 직접불 원칙(질의회시 관련)
- 회시 요지: 근로자가 임금 직접지급을 요구하는 경우 사용자의 지급 방식(예: 현금 지급 등)과 관련하여 임금 직접불 원칙 취지를 검토하는 취지의 회시로 제시됨.
- 사용자 사례와의 관련성: 사용자가 “제3자 지급/정산 방식”을 내세우며 지급을 미루는 경우, 직접불 원칙 관점에서 사용자 지급의무 및 지급 방식의 한계가 문제될 수 있음을 시사.

# 결론

- 현재 정보 기준 분석 결과: 제공된 조문/자료만으로는 화물차 일용직의 **임금 지급 책임의 ‘주체’를 단정할 수 없음**. 다만, (1) 해당 운송이 건설공사와 결합된 **건설 도급(2차 이상)** 구조로 평가되고, (2) 임금체불이 하수급인 소관 임금으로 보이는 경우에는 **직상 수급인도 하수급인과 연대하여 임금 지급 책임을 질 수 있음(근로기준법 제44조의2)**. 그 외의 경우에는 “사용자(고용관계의 실질을 가지는 자)” 개념과 계약구조에 따라 책임자가 달라질 여지가 큼.

- 주요 근거 조항 재정리 (법률명 + 조번호)
  - 근로기준법 제44조의2 제1항(연대책임: 건설공사 2차 이상 도급 + 하수급인 체불)
  - 근로기준법 제44조의2 제2항(직상 수급인 범위: 직상 수급인이 건설사업자가 아닌 경우 최하위 건설사업자)
  - 근로기준법 제43조의6 제1항·제3항(임금체불 책임자 특정 및 자료 제공 요청/의무)
  - 근로기준법 시행령 제2조 제1항(평균임금 산정 관련)
